# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbrahimAmr-PR/flyrank-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Selected Lane: Lane 2 — Refresh / Content Opportunity Scoring

ML Task Type: Binary Classification & Priority Scoring.
Classification: The model classifies each page as experiencing performance decline (1) or remaining stable/growing (0).
Scoring & Ranking: The model outputs predicted decay probabilities to generate a prioritized action list for content editors to optimize human bandwidth.

In [ ]:
import pandas as pd
from pathlib import Path

path = '/content/content_refresh_anonymized (1).csv'
df = pd.read_csv(path)
print("Task Type: Binary Classification & Scoring")
print(df['trend_direction'].value_counts())

Task Type: Binary Classification & Scoring
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Predicted Target: is_declining (Binary Proxy Label).
Target Source: Derived directly from observed performance data where is_declining = 1 if trend_direction == 'down', otherwise 0.
Proxy Justification: Serves as an observational proxy for pages actively losing organic search traffic and search engine visibility

In [ ]:
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print(f"Target column created: 'is_declining'")
print(f"Target distribution (Mean decay rate): {df['is_declining'].mean():.4f}")

Target column created: 'is_declining'
Target distribution (Mean decay rate): 0.5421


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metrics: Precision@K and PR-AUC (Precision-Recall AUC).

Why Precision@K? Editorial teams have limited capacity (e.g., refreshing 50–100 pages/week). High Precision in top recommendations ensures human hours aren't wasted on stable pages (minimizing Type I errors).

Why PR-AUC? Missed declining pages result in permanent traffic loss (Type II errors). Given the target ratio (~54% declining), PR-AUC is more actionable than standard Accuracy.

In [ ]:
total_count = len(df)
decay_count = df['is_declining'].sum()
print(f"Total instances: {total_count}")
print(f"Positives (Decaying): {decay_count}")
print(f"Random Baseline Precision: {decay_count / total_count:.4f}")

Total instances: 30000
Positives (Decaying): 16262
Random Baseline Precision: 0.5421


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: A single pseudonymized content item (one page).

Grain Verification: Each row represents unique content performance over observed evaluation windows.

In [ ]:
print(f"Unit of Analysis: 1 row = 1 unique content item (page)")
print(f"Total Rows: {len(df)}")
print(f"Unique Content IDs: {df['content_id'].nunique()}")
df[['content_id', 'search_volume', 'competition', 'trend_direction', 'is_declining']].head()

Unit of Analysis: 1 row = 1 unique content item (page)
Total Rows: 30000
Unique Content IDs: 30000


,content_id,search_volume,competition,trend_direction,is_declining
0,content_304f48230142,10.0,0.67,down,1
1,content_a1fb4e703a9e,90.0,0.01,down,1
2,content_9aa793d4d895,0.0,0.00,down,1
3,content_331d6c4de07b,10.0,0.00,stable,0
4,content_d99b7a2d90ca,0.0,0.00,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why Fixed Rules Fail:
Static heuristic rules (e.g., "refresh if page age > 12 months") fail to scale. They treat high-traffic core pages the same as low-value niche articles and cannot dynamically adapt to multi-variable interactions.

Why ML Wins:
ML captures complex non-linear interactions across search volume, competition, CPC, content intent, and directional trends simultaneously to dynamically score decay severity.

In [ ]:
feature_cols = ['search_volume', 'competition', 'cpc', 'is_declining']
print("Observed Feature Interactions Across Groups:")
print(df.groupby('trend_direction')[['search_volume', 'competition', 'cpc']].mean())

Observed Feature Interactions Across Groups:
                 search_volume  competition       cpc
trend_direction                                      
down                133.376490     0.144611  0.454844
flat                209.167483     0.165387  0.567140
new                  90.098039     0.186201  0.674183
stable              196.900070     0.129773  0.469222
up                  211.049534     0.163428  0.546758


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.